# 面试题：多模态检索怎样融合文本与图片，并做同口径评估？

## 可以直接复述的回答

多模态检索应把文本和图片各自编码、分别计算相似度，再在候选或分数层融合，不能假设两个通道的原始分值天然可比。文本 TF-IDF 擅长精确属性，图片向量擅长颜色、外观和品类；当查询文字只有“同款”时，视觉通道尤其关键。一个透明的教学实现可以手写 TF-IDF 向量、余弦相似度、每查询 min-max 归一化和加权融合。评估必须在同一商品全集、同一人工相关标签上比较 text-only 与 multimodal，避免各自挑有利样本。缺失图片、零向量和单通道常量分数都需要显式退化策略。原始文本分与 cosine 直接相加会造成量纲支配，本题会复现该错误并用归一化修正。数据包含 8 个商品和 6 个图文查询，逐项打印两个通道、融合分和最终排名。

## 真实案例

商品字段包含标题、人工分词和 8 维可解释图片向量，维度表示鞋、服装、包、杯、音频、红、蓝、黑等视觉概念。图片向量模拟离线视觉编码器产物，检索与融合为真实 NumPy 计算；教学向量不代表真实 CLIP 效果。

In [1]:
import math  # 导入对数函数以手写文本 IDF
from collections import Counter  # 导入计数器以统计文档频率
from pprint import pprint  # 导入结构化打印函数以展示候选账本
import numpy as np  # 导入 NumPy 以执行文本与图片向量计算
products = [{"id": "P1", "标题": "红色缓震跑鞋", "tokens": ["红色", "运动", "跑鞋", "缓震"], "image": [1.0, 0.0, 0.0, 0.0, 0.0, 0.9, 0.0, 0.0]}, {"id": "P2", "标题": "蓝色竞速跑鞋", "tokens": ["蓝色", "运动", "跑鞋", "竞速"], "image": [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.9, 0.0]}, {"id": "P3", "标题": "红色连衣裙", "tokens": ["红色", "服装", "连衣裙", "通勤"], "image": [0.0, 1.0, 0.0, 0.0, 0.0, 0.9, 0.0, 0.0]}, {"id": "P4", "标题": "黑色通勤双肩包", "tokens": ["黑色", "通勤", "双肩包", "收纳"], "image": [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.9]}, {"id": "P5", "标题": "猫咪陶瓷杯", "tokens": ["猫咪", "礼物", "陶瓷杯", "白色"], "image": [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]}, {"id": "P6", "标题": "黑色降噪耳机", "tokens": ["黑色", "音频", "耳机", "降噪"], "image": [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.9]}, {"id": "P7", "标题": "灰色布艺沙发", "tokens": ["灰色", "家具", "沙发", "客厅"], "image": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2]}, {"id": "P8", "标题": "桌面阅读灯", "tokens": ["桌面", "照明", "阅读灯", "护眼"], "image": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1]}]  # 构造八个带文本和视觉概念的商品
queries = [{"id": "Q1", "文字": "运动同款", "tokens": ["运动"], "image": [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0], "gold": "P2"}, {"id": "Q2", "文字": "红色同款", "tokens": ["红色"], "image": [0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], "gold": "P3"}, {"id": "Q3", "文字": "黑色同款", "tokens": ["黑色"], "image": [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0], "gold": "P6"}, {"id": "Q4", "文字": "猫咪礼物", "tokens": ["猫咪", "礼物"], "image": [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], "gold": "P5"}, {"id": "Q5", "文字": "灰色家具", "tokens": ["灰色", "家具"], "image": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2], "gold": "P7"}, {"id": "Q6", "文字": "桌面照明", "tokens": ["桌面", "照明"], "image": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1], "gold": "P8"}]  # 构造六个包含图像意图的查询
print("商品多模态字段预览：")  # 输出真实案例标题
pprint([{"id": product["id"], "标题": product["标题"], "tokens": product["tokens"], "image_vector": product["image"]} for product in products])  # 展示文本和图片向量输入
print("图文查询与人工相关商品：")  # 输出评估样本标题
pprint(queries)  # 展示六条查询的两个模态和标签

商品多模态字段预览：
[{'id': 'P1',
  'image_vector': [1.0, 0.0, 0.0, 0.0, 0.0, 0.9, 0.0, 0.0],
  'tokens': ['红色', '运动', '跑鞋', '缓震'],
  '标题': '红色缓震跑鞋'},
 {'id': 'P2',
  'image_vector': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.9, 0.0],
  'tokens': ['蓝色', '运动', '跑鞋', '竞速'],
  '标题': '蓝色竞速跑鞋'},
 {'id': 'P3',
  'image_vector': [0.0, 1.0, 0.0, 0.0, 0.0, 0.9, 0.0, 0.0],
  'tokens': ['红色', '服装', '连衣裙', '通勤'],
  '标题': '红色连衣裙'},
 {'id': 'P4',
  'image_vector': [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.9],
  'tokens': ['黑色', '通勤', '双肩包', '收纳'],
  '标题': '黑色通勤双肩包'},
 {'id': 'P5',
  'image_vector': [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
  'tokens': ['猫咪', '礼物', '陶瓷杯', '白色'],
  '标题': '猫咪陶瓷杯'},
 {'id': 'P6',
  'image_vector': [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.9],
  'tokens': ['黑色', '音频', '耳机', '降噪'],
  '标题': '黑色降噪耳机'},
 {'id': 'P7',
  'image_vector': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2],
  'tokens': ['灰色', '家具', '沙发', '客厅'],
  '标题': '灰色布艺沙发'},
 {'id': 'P8',
  'image_vector': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.

## Baseline / 基线：只使用文本 TF-IDF

文本基线用商品全集建立词表和 IDF，再把 query 与商品映射到同一空间。前三条查询只有宽泛属性，因此同分时按商品 ID 排序，会选错图片真正指向的款式。

In [2]:
vocabulary = sorted({token for product in products for token in product["tokens"]})  # 从商品文本构造稳定词表
token_to_index = {token: index for index, token in enumerate(vocabulary)}  # 建立 token 到向量列号的映射
document_frequency = Counter()  # 创建商品文档频率计数器
for product in products:  # 遍历全部商品文本
    for token in set(product["tokens"]):  # 单个商品内同词只计一次
        document_frequency[token] += 1  # 累加包含当前词的商品数
def tfidf_vector(tokens):  # 定义手写 TF-IDF 向量化函数
    frequencies = Counter(tokens)  # 统计输入文本词频
    vector = np.zeros(len(vocabulary), dtype=np.float64)  # 创建词表维度的零向量
    for token, frequency in frequencies.items():  # 遍历输入中出现的词
        if token in token_to_index:  # 仅编码商品词表中已知词
            idf = math.log((len(products) + 1) / (document_frequency[token] + 1)) + 1  # 计算平滑 IDF
            vector[token_to_index[token]] = frequency * idf  # 写入 TF 乘 IDF 权重
    return vector  # 返回可比较的文本向量
def cosine(left, right):  # 定义两个 NumPy 向量的余弦相似度
    denominator = np.linalg.norm(left) * np.linalg.norm(right)  # 计算范数乘积
    return float(left @ right / denominator) if denominator else 0.0  # 计算归一化点积并保护零向量
product_text_vectors = {product["id"]: tfidf_vector(product["tokens"]) for product in products}  # 预计算商品文本向量
def text_ranking(query):  # 定义 text-only 候选排序
    query_vector = tfidf_vector(query["tokens"])  # 计算查询文本向量
    rows = [{"product_id": product["id"], "text_score": cosine(query_vector, product_text_vectors[product["id"]])} for product in products]  # 实算所有商品文本相似度
    return sorted(rows, key=lambda row: (-row["text_score"], row["product_id"]))  # 返回稳定文本排名
baseline_rankings = {query["id"]: text_ranking(query) for query in queries}  # 对六条查询运行文本基线
baseline_hits = sum(baseline_rankings[query["id"]][0]["product_id"] == query["gold"] for query in queries)  # 统计文本基线 Top1 命中
print("Text-only Baseline Top3：")  # 输出基线结果标题
pprint([{"查询": query["文字"], "gold": query["gold"], "Top3": [(row["product_id"], round(row["text_score"], 3)) for row in baseline_rankings[query["id"]][:3]]} for query in queries])  # 展示每条查询的文本候选
print(f"Text-only Top1={baseline_hits}/{len(queries)}")  # 输出基线汇总指标

Text-only Baseline Top3：
[{'Top3': [('P1', 0.475), ('P2', 0.454), ('P3', 0.0)],
  'gold': 'P2',
  '查询': '运动同款'},
 {'Top3': [('P1', 0.475), ('P3', 0.454), ('P2', 0.0)],
  'gold': 'P3',
  '查询': '红色同款'},
 {'Top3': [('P4', 0.454), ('P6', 0.436), ('P1', 0.0)],
  'gold': 'P6',
  '查询': '黑色同款'},
 {'Top3': [('P5', 0.707), ('P1', 0.0), ('P2', 0.0)],
  'gold': 'P5',
  '查询': '猫咪礼物'},
 {'Top3': [('P7', 0.707), ('P1', 0.0), ('P2', 0.0)],
  'gold': 'P7',
  '查询': '灰色家具'},
 {'Top3': [('P8', 0.707), ('P1', 0.0), ('P2', 0.0)],
  'gold': 'P8',
  '查询': '桌面照明'}]
Text-only Top1=3/6


## 图片相似度与逐查询归一化

图片通道直接对查询图片向量与商品图片向量做 cosine。融合前分别对当前查询的文本分和图片分做 min-max；当一个通道全部相等时返回全零，避免除零和虚假区分。

In [3]:
def minmax(values):  # 定义每查询分数归一化函数
    array = np.asarray(values, dtype=np.float64)  # 把分数列表转换为 NumPy 数组
    minimum = float(array.min())  # 读取当前通道最小分
    maximum = float(array.max())  # 读取当前通道最大分
    return (array - minimum) / (maximum - minimum) if maximum > minimum else np.zeros_like(array)  # 在有区分度时缩放到零一范围
def multimodal_ranking(query, text_weight=0.4, image_weight=0.6):  # 定义双通道归一化融合排序
    query_text_vector = tfidf_vector(query["tokens"])  # 计算查询文本 TF-IDF 向量
    query_image_vector = np.asarray(query["image"], dtype=np.float64)  # 读取查询图片概念向量
    text_scores = [cosine(query_text_vector, product_text_vectors[product["id"]]) for product in products]  # 计算全部文本相似度
    image_scores = [cosine(query_image_vector, np.asarray(product["image"], dtype=np.float64)) for product in products]  # 计算全部图片相似度
    normalized_text = minmax(text_scores)  # 把文本分数校准到零一范围
    normalized_image = minmax(image_scores)  # 把图片分数校准到零一范围
    rows = []  # 创建逐商品融合账本
    for index, product in enumerate(products):  # 遍历八个候选商品
        fused_score = text_weight * normalized_text[index] + image_weight * normalized_image[index]  # 按可解释权重组合两个模态
        rows.append({"product_id": product["id"], "text_raw": round(text_scores[index], 4), "image_raw": round(image_scores[index], 4), "text_norm": round(float(normalized_text[index]), 4), "image_norm": round(float(normalized_image[index]), 4), "fused": round(float(fused_score), 4)})  # 保存原始分、校准分和融合分
    return sorted(rows, key=lambda row: (-row["fused"], row["product_id"]))  # 返回融合后的稳定排名
multimodal_rankings = {query["id"]: multimodal_ranking(query) for query in queries}  # 对六条查询运行图文检索
print("Q2 红色同款的双通道候选账本：")  # 输出关键融合过程标题
pprint(multimodal_rankings["Q2"])  # 展示文本歧义如何被图片品类消除

Q2 红色同款的双通道候选账本：
[{'fused': 0.9821,
  'image_norm': 1.0,
  'image_raw': 0.9986,
  'product_id': 'P3',
  'text_norm': 0.9553,
  'text_raw': 0.4542},
 {'fused': 0.6842,
  'image_norm': 0.4737,
  'image_raw': 0.473,
  'product_id': 'P1',
  'text_norm': 1.0,
  'text_raw': 0.4755},
 {'fused': 0.0,
  'image_norm': 0.0,
  'image_raw': 0.0,
  'product_id': 'P2',
  'text_norm': 0.0,
  'text_raw': 0.0},
 {'fused': 0.0,
  'image_norm': 0.0,
  'image_raw': 0.0,
  'product_id': 'P4',
  'text_norm': 0.0,
  'text_raw': 0.0},
 {'fused': 0.0,
  'image_norm': 0.0,
  'image_raw': 0.0,
  'product_id': 'P5',
  'text_norm': 0.0,
  'text_raw': 0.0},
 {'fused': 0.0,
  'image_norm': 0.0,
  'image_raw': 0.0,
  'product_id': 'P6',
  'text_norm': 0.0,
  'text_raw': 0.0},
 {'fused': 0.0,
  'image_norm': 0.0,
  'image_raw': 0.0,
  'product_id': 'P7',
  'text_norm': 0.0,
  'text_raw': 0.0},
 {'fused': 0.0,
  'image_norm': 0.0,
  'image_raw': 0.0,
  'product_id': 'P8',
  'text_norm': 0.0,
  'text_raw': 0.0}]


## 同一评测集结果与结果解读

后三条查询的文字已经足够明确，两种方法都应答对；前三条存在文本并列，视觉向量把蓝鞋、红裙和黑耳机分别提升到第一。逐查询表既显示收益，也防止平均指标掩盖退化样本。

In [4]:
comparison_rows = []  # 创建逐查询同口径结果表
for query in queries:  # 遍历六条人工标注查询
    text_top = baseline_rankings[query["id"]][0]["product_id"]  # 读取 text-only 第一名
    multimodal_top = multimodal_rankings[query["id"]][0]["product_id"]  # 读取融合第一名
    comparison_rows.append({"查询": query["文字"], "gold": query["gold"], "Text Top1": text_top, "Multimodal Top1": multimodal_top, "文本正确": text_top == query["gold"], "融合正确": multimodal_top == query["gold"]})  # 保存两种方法逐样本决策
multimodal_hits = sum(row["融合正确"] for row in comparison_rows)  # 统计融合 Top1 命中
text_mrr = sum(1 / ([row["product_id"] for row in baseline_rankings[query["id"]]].index(query["gold"]) + 1) for query in queries) / len(queries)  # 计算文本基线 MRR
multimodal_mrr = sum(1 / ([row["product_id"] for row in multimodal_rankings[query["id"]]].index(query["gold"]) + 1) for query in queries) / len(queries)  # 计算融合方案 MRR
print("逐查询同候选集结果：")  # 输出结果表标题
pprint(comparison_rows)  # 展示每条查询是否被视觉通道修正
print(f"Top1 从 {baseline_hits}/{len(queries)} 到 {multimodal_hits}/{len(queries)}，MRR 从 {text_mrr:.4f} 到 {multimodal_mrr:.4f}")  # 输出同评测集汇总指标

逐查询同候选集结果：
[{'Multimodal Top1': 'P2',
  'Text Top1': 'P1',
  'gold': 'P2',
  '文本正确': False,
  '查询': '运动同款',
  '融合正确': True},
 {'Multimodal Top1': 'P3',
  'Text Top1': 'P1',
  'gold': 'P3',
  '文本正确': False,
  '查询': '红色同款',
  '融合正确': True},
 {'Multimodal Top1': 'P6',
  'Text Top1': 'P4',
  'gold': 'P6',
  '文本正确': False,
  '查询': '黑色同款',
  '融合正确': True},
 {'Multimodal Top1': 'P5',
  'Text Top1': 'P5',
  'gold': 'P5',
  '文本正确': True,
  '查询': '猫咪礼物',
  '融合正确': True},
 {'Multimodal Top1': 'P7',
  'Text Top1': 'P7',
  'gold': 'P7',
  '文本正确': True,
  '查询': '灰色家具',
  '融合正确': True},
 {'Multimodal Top1': 'P8',
  'Text Top1': 'P8',
  'gold': 'P8',
  '文本正确': True,
  '查询': '桌面照明',
  '融合正确': True}]
Top1 从 3/6 到 6/6，MRR 从 0.7500 到 1.0000


## 失败案例：原始分直接相加与缺失模态

下面构造“红色运动同款”但图片明确是连衣裙的冲突查询，再把文本的未归一化命中权重与 image cosine 相加，使文本尺度压过图片而选中红鞋。修正使用每查询校准；如果没有图片，则把 image 权重置零并重新归一权重，而不是用零向量伪装成真实图片。

In [5]:
failure_query = {"tokens": ["红色", "运动"], "image": queries[1]["image"]}  # 构造文字误导为运动但图片明确指向裙子的冲突查询
unsafe_rows = []  # 创建量纲不一致的错误融合表
for product in products:  # 遍历八个商品候选
    exact_weight = 3.0 * sum(token in product["tokens"] for token in failure_query["tokens"])  # 构造零到三的未归一化文本命中分
    image_score = cosine(np.asarray(failure_query["image"], dtype=np.float64), np.asarray(product["image"], dtype=np.float64))  # 计算零到一的图片余弦分
    unsafe_rows.append({"product_id": product["id"], "text_raw": exact_weight, "image_raw": round(image_score, 4), "raw_sum": round(exact_weight + image_score, 4)})  # 直接相加两个不同量纲分数
unsafe_rows = sorted(unsafe_rows, key=lambda row: (-row["raw_sum"], row["product_id"]))  # 对错误融合分排序
fixed_failure_ranking = multimodal_ranking(failure_query)  # 用逐通道归一化处理同一冲突查询
missing_image_query = {"tokens": ["猫咪", "礼物"], "image": [0.0] * 8}  # 构造用户没有上传有效图片的查询
fallback_ranking = multimodal_ranking(missing_image_query, text_weight=1.0, image_weight=0.0)  # 在缺图时明确退化为文本检索
print("失败案例：未校准原始分 Top3")  # 输出错误融合标题
pprint(unsafe_rows[:3])  # 展示文本尺度造成的错误候选
print("修正后的冲突查询第一名：", fixed_failure_ranking[0])  # 展示归一化融合恢复红裙
print("缺失图片时的 text-only 回退第一名：", fallback_ranking[0])  # 展示缺图门禁不会除零或随机排序

失败案例：未校准原始分 Top3
[{'image_raw': 0.473, 'product_id': 'P1', 'raw_sum': 6.473, 'text_raw': 6.0},
 {'image_raw': 0.9986, 'product_id': 'P3', 'raw_sum': 3.9986, 'text_raw': 3.0},
 {'image_raw': 0.0, 'product_id': 'P2', 'raw_sum': 3.0, 'text_raw': 3.0}]
修正后的冲突查询第一名： {'product_id': 'P3', 'text_raw': 0.3212, 'image_raw': 0.9986, 'text_norm': 0.4776, 'image_norm': 1.0, 'fused': 0.7911}
缺失图片时的 text-only 回退第一名： {'product_id': 'P5', 'text_raw': 0.7071, 'image_raw': 0.0, 'text_norm': 1.0, 'image_norm': 0.0, 'fused': 1.0}


## 生产差距

线上系统需要真实文本与视觉 encoder、向量版本管理、ANN 召回、跨模态 hard negatives 和模态缺失监控。分数校准应按查询类型或学习排序训练，图片还涉及版权、安全审核和对抗内容；评估需覆盖文本主导、图片主导、冲突模态及冷启动，并监控延迟和索引新鲜度。

In [6]:
assert len(products) == 8 and len(queries) == 6  # 验证案例包含八个商品和六条图文查询
assert baseline_hits == 3  # 验证文本基线在三条视觉歧义查询上失败
assert multimodal_hits == len(queries)  # 验证归一化融合命中全部人工相关商品
assert multimodal_mrr > text_mrr  # 验证同一候选集上的平均排名得到改善
assert unsafe_rows[0]["product_id"] == "P1"  # 验证原始分相加仍被红色文本支配
assert fixed_failure_ranking[0]["product_id"] == "P3"  # 验证校准融合使用图片品类修正冲突查询
assert fallback_ranking[0]["product_id"] == "P5"  # 验证缺失图片时正确退化到文本结果
print("最小回归测试通过：双模态向量、分数校准、同口径评估与缺图回退均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：双模态向量、分数校准、同口径评估与缺图回退均满足预期
